# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 3: Optimización de Recursos: CPU, Memoria y Red

### Objetivo:
Aprender a optimizar el consumo de recursos (procesamiento asíncrono en CPU y manejo de caché en memoria) para aumentar la capacidad del sistema de agentes y disminuir los costos operacionales de red.

### Optimización de Recursos en Producción:
1. **CPU Optimization:** Implementar procesamiento en paralelo de múltiples requests para no bloquear el hilo de ejecución principal.
2. **Memory Management:** Almacenar en caché las respuestas de cómputo costoso o llamadas al LLM para no repetirlas si la consulta del usuario es idéntica.
3. **Network Optimization:** Reducir el consumo de red mediante connection pooling hacia APIs externas, compresión de payloads y batching.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Optimización de CPU con Asyncio y Threads
Utilizaremos un `ThreadPoolExecutor` acoplado a funciones asíncronas para disparar un lote de consultas de forma paralela en el agente de Wikipedia real, evitando el procesamiento bloqueante secuencial.


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import time

class OptimizedAgent:
    def __init__(self, executor):
        self.executor = executor
        # Configurar un pool de hilos para llamadas bloqueantes
        self.thread_pool = ThreadPoolExecutor(max_workers=4)
    
    def process_single(self, req):
        try:
            if llm is None:
                time.sleep(0.3)
                return f"Simulado: {req}"
            res = self.executor.invoke({"input": req})
            return res.get("output", "")
        except Exception as e:
            return f"Error: {e}"

    async def process_batch(self, requests):
        loop = asyncio.get_event_loop()
        # Crear tareas concurrentes en el pool de hilos
        tasks = [
            loop.run_in_executor(self.thread_pool, self.process_single, req)
            for req in requests
        ]
        # Esperar la ejecución de todo el lote de forma no bloqueante
        return await asyncio.gather(*tasks)

# Lote de consultas concurrentes
requests_batch = [
    "¿Quién fue Albert Einstein?",
    "¿Qué es la fotosíntesis?",
    "Busca información sobre las ballenas azules.",
    "Dame datos del Imperio Inca."
]

agent_opt = OptimizedAgent(agent_executor)

start = time.time()
print("Procesando lote de consultas concurrentes...")

# Ejecutar lote en el bucle de eventos asíncrono
loop = asyncio.get_event_loop()
results = loop.run_until_complete(agent_opt.process_batch(requests_batch))
end = time.time()

for req, res in zip(requests_batch, results):
    print(f"\nConsulta: '{req}'")
    print(f"Respuesta: {res[:120]}...")

print(f"\nTiempo total de ejecución en paralelo: {end - start:.3f} segundos")


### Gestión de Memoria mediante Caché LRU (Least Recently Used)
Crearemos una clase envoltoria `MemoryEfficientAgent` que implementa una caché en memoria para respuestas del agente. Si una consulta idéntica ya fue respondida, no volverá a llamar al modelo real ni al scraping de Wikipedia, optimizando memoria, latencia y costos operacionales.


In [ ]:
class MemoryEfficientAgent:
    def __init__(self, executor, max_cache_size=5):
        self.executor = executor
        self.cache = {}
        self.max_cache_size = max_cache_size

    def process_with_cache(self, input_data):
        cache_key = hash(input_data)
        
        # Verificar si existe en la memoria caché
        if cache_key in self.cache:
            print(f"[Cache HIT] Entrada recuperada de memoria para: '{input_data}'")
            return self.cache[cache_key]
        
        # Si no existe, invocar al modelo real
        print(f"[Cache MISS] Invocando agente real para: '{input_data}'")
        try:
            if llm is None:
                result = f"Respuesta simulada para {input_data}"
            else:
                response = self.executor.invoke({"input": input_data})
                result = response.get("output", "")
        except Exception as e:
            result = f"Error: {e}"
        
        # Controlar el tamaño del caché con política LRU simple (eliminar primer elemento insertado)
        if len(self.cache) >= self.max_cache_size:
            oldest_key = next(iter(self.cache))
            print(f"[Cache EVICT] Removiendo entrada antigua del caché.")
            del self.cache[oldest_key]
            
        self.cache[cache_key] = result
        return result

# Instanciar agente con caché de tamaño 3
agent_mem = MemoryEfficientAgent(agent_executor, max_cache_size=3)

print("--- Consulta 1 (MISS) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))

print("\n--- Consulta 2 (HIT - Repetida) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))

print("\n--- Consultas 3 y 4 (MISS - Llenado caché) ---")
agent_mem.process_with_cache("¿Qué es un electrón?")
agent_mem.process_with_cache("Busca datos de los dinosaurios.")

print("\n--- Consulta 5 (MISS - Provocará EVICT de Galileo) ---")
agent_mem.process_with_cache("¿Qué es la Vía Láctea?")

print("\n--- Consulta 6 (MISS de nuevo - Galileo fue eliminado) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))


### Preguntas de Análisis
1. **¿De qué manera la optimización de latencia mediante concurrencia impacta en el costo operativo de procesamiento en lote de prompts?**
2. **¿Qué riesgos e implicaciones de seguridad/privacidad tiene almacenar en caché las respuestas de los usuarios?**
3. **Mencione 3 técnicas adicionales de optimización de red aplicables a la arquitectura de agentes.**
